# run_training — Lanzar los 3 modelos en SageMaker
Ejecutar en el notebook instance `semi2-ml`. Cada `SKLearn` estimator entrena un script leyendo el ABT desde S3 y sube el `model.tar.gz` a `models/`.

In [ ]:
import sagemaker
from sagemaker.sklearn import SKLearn

sess = sagemaker.Session()
ROLE = 'arn:aws:iam::253685958632:role/SageMakerExecutionRole-Semi2'
BUCKET = 's3://mortality-analytics-semi2'
FW = '1.2-1'   # version de scikit-learn del contenedor
INSTANCE = 'ml.m5.large'   # free tier para training

## Modelo A — Exceso (Regresión Lineal)

In [ ]:
est_a = SKLearn(entry_point='linear_excess.py', source_dir='../../ml',
               role=ROLE, instance_type=INSTANCE, framework_version=FW,
               output_path=f'{BUCKET}/models/exceso/')
est_a.fit({'train': f'{BUCKET}/abt/ine/'})

## Modelo B — Random Forest

In [ ]:
est_b = SKLearn(entry_point='rf_mortality.py', source_dir='../../ml',
               role=ROLE, instance_type=INSTANCE, framework_version=FW,
               output_path=f'{BUCKET}/models/rf/')
est_b.fit({'train': f'{BUCKET}/abt/ine/'})

## Modelo C — K-means (corre pre y post)

In [ ]:
for etiqueta in ['pre', 'post']:
    est_c = SKLearn(entry_point='kmeans_segments.py', source_dir='../../ml',
                   role=ROLE, instance_type=INSTANCE, framework_version=FW,
                   hyperparameters={'k': 4},
                   output_path=f'{BUCKET}/models/kmeans/{etiqueta}/')
    est_c.fit({'train': f'{BUCKET}/abt/ine_wide/{etiqueta}/'})

## ⚠️ Al terminar, apaga el notebook para no gastar
En la consola SageMaker → Notebook instances → `semi2-ml` → **Stop**. Apagado = $0.